**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Modern 3D Vision Studio

**Lessico prima dell’avvio.** NeRF significa Neural Radiance Field; 3DGS, 3D Gaussian Splatting; VGGT, Visual Geometry Grounded Transformer. Rendering calcola un’immagine dalla rappresentazione della scena e dalla camera. I calcoli qui sono analitici e non eseguono questi tre modelli.

Computer Vision · Lezione 20

Geometry audit con scene sintetiche assegnate. I calcoli spiegano e verificano i meccanismi; non sono inferenze NeRF, 3DGS o VGGT.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

<!-- C02 -->
## Avvio in Colab e uso dei materiali
Aprire questo notebook da File > Carica notebook. Eseguire C03 **nello stesso runtime** usato nello studio. Il percorso principale è autonomo: genera i dati, non scarica pesi e funziona su CPU. Il pacchetto ZIP contiene anche riferimenti e recupero, ma non è necessario estrarlo per eseguire questo notebook. Tre TODO: C07, C15, C23. Salvare notebook e output prima che la VM termini. Le celle Cxx contano anche quelle di testo. Leggere N01–N10 e rispondere nel report.

In [ ]:
# C03
import sys, importlib.util, subprocess, json, time, platform, hashlib, zipfile
from pathlib import Path
for package in ['numpy', 'matplotlib']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
import numpy as np
import matplotlib.pyplot as plt
OUT=Path('output_lezione20'); OUT.mkdir(exist_ok=True)
START=time.perf_counter()
metrics={'provenance':'esperimenti analitici su dati sintetici assegnati; nessuna inferenza NeRF/3DGS/VGGT', 'device':'CPU', 'numpy':np.__version__, 'python':platform.python_version()}
print('Setup CPU nello stesso runtime. Output:', OUT.resolve())

<!-- C04 -->
## N01 · Scena e convenzioni

**Richiamo di coordinate.** K contiene intrinseci in pixel; R e t trasformano mondo in camera; il centro C è espresso nel mondo. La depth è la coordinata Z della camera, il range la distanza euclidea da C. La scala della scena non è automaticamente metrica. Le formule usano colonne, il codice punti per riga.

Le coordinate hanno unità arbitrarie. K usa pixel, R, t trasformano mondo in camera. L’immagine ha 640×480 pixel, x verso destra, y verso il basso, z in avanti. Quale origine esprime t? Quale esprime il centro camera?

In [ ]:
# C05
K=np.array([[500.,0,320],[0,500,240],[0,0,1]])
R=np.eye(3); t=np.array([-1.,0,0])
X=np.array([[0.,0,4],[1.,0,4],[0.,1,4],[1.,1,6],[-1.,-.5,5]])
C=-R.T@t
print('Centro camera mondo:',C, 'Traslazione mondo-camera:',t)

<!-- C06 -->
## N02 · Proiezione

**Prima della proiezione.** Pinhole indica il modello prospettico ideale; fx, fy sono focali in pixel e cx, cy il principal point. Si calcola il punto in camera prima della divisione per Z>0. il punto (2, 0, 4) ha depth 4 e range √20≈4.472136. Lo schema è concettuale non in scala.

Completare C07. Applicare Xc=R X+t, poi K e divisione per z. Per semplicità tutti i punti hanno z positivo. La funzione deve rifiutare punti con z≤0. Quale sarebbe il problema di usare C al posto di t?

In [ ]:
# C07
def project(X, K, R, t):
    # TODO: completare dopo la discussione sulle slide.
    raise NotImplementedError("Completare TODO 07")

In [ ]:
# C08
uv=project(X,K,R,t)
assert np.allclose(uv[0],[195,240])
# A non-identity rotation detects R vs R.T errors.
Rcheck=np.array([[0.,-1,0],[1,0,0],[0,0,1]])
assert np.allclose(project(np.array([[1.,2,4]]),K,Rcheck,np.zeros(3)),[[70.,365.]])
try:
    project(np.array([[0.,0,-1.]]),K,R,np.zeros(3))
    raise AssertionError('La funzione deve rifiutare z negativo')
except ValueError:
    pass
metrics['projection_first_pixel']=uv[0].tolist()
fig,ax=plt.subplots(figsize=(8,5)); ax.scatter(*uv.T,s=70)
for i,p in enumerate(uv): ax.annotate(str(i),p+3)
ax.set(xlim=(0,640),ylim=(480,0),xlabel='x [pixel]',ylabel='y [pixel]',title='Punti sintetici, camera 2'); ax.grid(); fig.savefig(OUT/'proiezione.png',dpi=150,bbox_inches='tight'); plt.show()
print(uv)

<!-- C09 -->
## N03 · Ambiguità di scala
Moltiplicare punti e centro camera per tre. Perché la proiezione rimane identica? L’unità arbitraria può essere chiamata metro senza altre informazioni?

In [ ]:
# C10
uv_scaled=project(3*X,K,R,3*t)
scale_error=float(np.max(np.abs(uv_scaled-uv)))
assert scale_error<1e-10
metrics['scale_reprojection_max_px']=scale_error
print('Differenza massima di pixel dopo scala ×3:',scale_error)

<!-- C11 -->
## N04 · Raggio e profondità

**Due parametrizzazioni.** Depth significa Zc, range la norma del punto in camera. K⁻¹p ha componente Z=1; moltiplicarlo per depth ricostruisce il punto. Dopo normalizzazione la direzione è unitaria: il moltiplicatore lungo quel raggio indica invece range. In r(t), t è uno scalare, distinto dalla traslazione vettoriale t delle pose.

Il vettore K⁻¹[x, y, 1] ha componente z=1. Se lo normalizziamo, il parametro lungo il raggio misura range euclideo. Calcolare il punto del pixel (570, 240) con depth z=4 e confrontare range.

In [ ]:
# C12
pixel=np.array([570.,240.,1.]); v=np.linalg.solve(K,pixel)
Xcam=4*v; ray=v/np.linalg.norm(v); range_value=float(np.linalg.norm(Xcam))
assert np.allclose(Xcam,[2,0,4])
metrics['depth_vs_range']={'depth_z':4.,'range':range_value,'point_camera':Xcam.tolist()}
print('v:',v,'d unitario:',ray,'z:',4.,'range:',range_value)

<!-- C13 -->
## N05 · Rendering volumetrico

**Prima della quadratura.** σ è densità di estinzione non negativa, δ lunghezza dell’intervallo, α la sua opacità condizionata all’arrivo del raggio. T è trasmittanza residua. Esclusiva significa prodotto dei soli intervalli precedenti, con T₁=1. I pesi sono w=T·α; il fondo conserva la trasmittanza rimasta. Quadratura significa approssimare l’integrale con una somma pesata.

C14 assegna tre intervalli con densità costante. C15 deve calcolare α, trasmittanza esclusiva e pesi. Con fondo bianco, la parte non assorbita contribuisce al colore. Nessuna rete viene addestrata.

In [ ]:
# C14
sigma=np.log(2)*np.ones(3); delta=np.ones(3)
colors=np.eye(3); bg=np.ones(3)
print('Densità:',sigma,'Lunghezze:',delta)

In [ ]:
# C15
def volume_weights(sigma,delta):
    # TODO: completare dopo la discussione sulle slide.
    raise NotImplementedError("Completare TODO 15")

In [ ]:
# C16
alpha,T,w=volume_weights(sigma,delta)
remaining=float(np.prod(1-alpha)); rgb=w@colors+remaining*bg
assert np.allclose(w,[.5,.25,.125]) and np.allclose(rgb,[.625,.375,.25])
metrics['volume']={'alpha':alpha.tolist(),'T':T.tolist(),'weights':w.tolist(),'background_weight':remaining,'rgb_white_background':rgb.tolist()}
fig,ax=plt.subplots(figsize=(8,4)); ax.bar(['rosso','verde','blu','sfondo'],list(w)+[remaining],color=['#c64547','#087f8c','#173d6c','#929a9c']); ax.set(ylabel='Peso',ylim=(0,1),title='Contributi lungo il raggio'); fig.savefig(OUT/'pesi_volume.png',dpi=150,bbox_inches='tight'); plt.show()
print(json.dumps(metrics['volume'],indent=2))

<!-- C17 -->
## Campionamento stratificato

**Lessico del campionamento.** Midpoint è il punto medio di ciascun intervallo. Coarse/fine distingue stima grossolana e campionamento più dettagliato; importance sampling concentra campioni dove i pesi della prima stima indicano contributi rilevanti. Il picco analitico di C18 non è un training NeRF.

La densità è un picco stretto fra due campioni grossolani. Confrontare midpoint quadrature con l’integrale numerico molto fine. La maggiore risoluzione migliora il caso scelto, senza garanzia per una qualunque scena. Il campionamento gerarchico di NeRF usa i pesi coarse, non la densità GT del nostro test.

In [ ]:
# C18
def narrow_density(x): return 12*np.exp(-.5*((x-.51)/.018)**2)
x_ref=(np.arange(100000)+.5)/100000
opacity_ref=float(1-np.exp(-np.mean(narrow_density(x_ref))))
sweep=[]
for count in [4,8,16,32,64,128]:
    x=(np.arange(count)+.5)/count
    opacity=float(1-np.exp(-np.mean(narrow_density(x))))
    sweep.append([count,opacity,abs(opacity-opacity_ref)])
metrics['sampling']={'reference_opacity':opacity_ref,'sweep':sweep}
print('Riferimento numerico:',opacity_ref); print(np.array(sweep))

<!-- C19 -->
## N06 · Covarianza 3D e proiezione

**Forma della primitiva.** Covarianza qui descrive estensione e orientamento spaziale della Gaussiana, non incertezza misurata del punto. Anisotropo significa estensione diversa per direzione. Q orienta la primitiva; Rc orienta mondo in camera. Le loro funzioni sono distinte.

Una Gaussiana ha centro μ, scale positive e rotazione Q. La covarianza Σ=Q diag(s²) Qᵀ è positiva. Lo Jacobiano della prospettiva va valutato nel centro in camera. Quali unità ha Σ2D?

In [ ]:
# C20
theta=np.pi/6; Q=np.array([[np.cos(theta),-np.sin(theta),0],[np.sin(theta),np.cos(theta),0],[0,0,1.]])
s=np.array([.2,.08,.12]); Sigma=Q@np.diag(s*s)@Q.T
mu=np.array([.5,.2,4.]); Rc=np.eye(3)
assert np.all(np.linalg.eigvalsh(Sigma)>0)
print('Autovalori 3D:',np.linalg.eigvalsh(Sigma))

<!-- C21 -->
## Jacobiano locale

**Prima dello Jacobiano.** J è la matrice 2×3 delle derivate locali della proiezione rispetto alle coordinate del centro in camera. La covarianza si ruota in camera e poi si proietta con J. Il footprint è l’estensione risultante sul piano immagine. Mahalanobis normalizza gli spostamenti usando scale e orientamento della covarianza.

Le derivate di u=fx X/Z+cx e v=fy Y/Z+cy danno una matrice 2×3. Il modello è locale: un’ellisse grande vicino alla camera può violare l’approssimazione. C22 confronta derivata analitica e differenze finite.

In [ ]:
# C22
def jacobian_perspective(mu,K):
    x,y,z=mu; fx,fy=K[0,0],K[1,1]
    return np.array([[fx/z,0,-fx*x/z**2],[0,fy/z,-fy*y/z**2]])
J=jacobian_perspective(mu,K)
eps=1e-6
p=lambda m: (K@m)[:2]/m[2]
J_fd=np.column_stack([(p(mu+eps*np.eye(3)[i])-p(mu-eps*np.eye(3)[i]))/(2*eps) for i in range(3)])
assert np.allclose(J,J_fd,atol=1e-6)
metrics['jacobian_max_error']=float(np.max(np.abs(J-J_fd)))
print(J)

In [ ]:
# C23
def project_covariance(Sigma,Rc,J):
    # TODO: completare dopo la discussione sulle slide.
    raise NotImplementedError("Completare TODO 23")

In [ ]:
# C24
Sigma2=project_covariance(Sigma,Rc,J)
eigen=np.linalg.eigvalsh(Sigma2)
assert Sigma2.shape==(2,2) and np.all(eigen>0)
# Nontrivial rotation plus diagonal covariance gives a known result.
Rcheck=np.array([[0.,-1,0],[1,0,0],[0,0,1]])
Jcheck=np.array([[2.,0,0],[0,3.,0]])
assert np.allclose(project_covariance(np.diag([1.,4,9]),Rcheck,Jcheck),np.diag([16.,9]))
metrics['gaussian']={'covariance_3d':Sigma.tolist(),'J':J.tolist(),'covariance_2d_px2':Sigma2.tolist(),'eigenvalues_2d':eigen.tolist()}
angle=np.linspace(0,2*np.pi,300); vals,vecs=np.linalg.eigh(Sigma2); ellipse=vecs@np.diag(np.sqrt(vals))@np.array([np.cos(angle),np.sin(angle)])
fig,ax=plt.subplots(figsize=(7,5)); ax.plot(ellipse[0],ellipse[1],lw=3); ax.set(xlabel='Δu [pixel]',ylabel='Δv [pixel]',title='Contorno di Mahalanobis 1 della Gaussiana proiettata'); ax.axis('equal'); ax.grid(); fig.savefig(OUT/'ellisse.png',dpi=150,bbox_inches='tight'); plt.show()
print('Σ2D [pixel²]:',Sigma2)

<!-- C25 -->
## N07 · Ordine di compositing

**Prima del compositing.** Splat è il contributo sfumato di una primitiva sui pixel. Compositing combina contributi ordinati dal vicino al lontano. α è l’opacità effettiva al pixel; non si sommano soltanto i colori e non si riordinano per ampiezza dello score.

Due splat al pixel hanno α=.6 e .5, colori rosso e blu. Fondo nero. Confrontare rosso davanti e blu davanti. Perché una somma commutativa non può modellare questa occlusione?

In [ ]:
# C26
def composite(a,c,bg=np.zeros(3)):
    trans=np.concatenate([[1.],np.cumprod(1-a)[:-1]])
    return (trans*a)@c + np.prod(1-a)*bg
rgb_rb=composite(np.array([.6,.5]),np.array([[1.,0,0],[0,0,1.]]))
rgb_br=composite(np.array([.5,.6]),np.array([[0.,0,1],[1.,0,0]]))
assert np.allclose(rgb_rb,[.6,0,.2]) and np.allclose(rgb_br,[.3,0,.5])
metrics['compositing']={'red_front':rgb_rb.tolist(),'blue_front':rgb_br.tolist()}
print('Rosso davanti:',rgb_rb,'Blu davanti:',rgb_br)

<!-- C27 -->
## N08 · Depth, point map e riferimento

**Output e riferimento.** Una depth map contiene Zc per pixel; una point map contiene un punto 3D per pixel nel riferimento dichiarato. Backprojection inverte la proiezione usando depth e K; l’inversa degli estrinseci porta il punto al riferimento comune. Il roundtrip nella stessa camera verifica una coerenza interna, non la scala esterna.

Costruire una depth map sintetica, fare backprojection e trasformare in mondo con Rᵀ(Xc−t). Riprojettare con la stessa camera verifica la convenzione; un errore nullo non prova che la profondità sia quella della scena reale. Non chiamare questi array output VGGT.

In [ ]:
# C28
grid_uv=np.array([[200.,180.],[320.,240.],[450.,300.]])
z=np.array([4.,5.,6.])
hom=np.column_stack([grid_uv,np.ones(3)])
Xc=(np.linalg.inv(K)@hom.T).T*z[:,None]
point_map=(Xc-t)@R
back_uv=project(point_map,K,R,t)
assert np.allclose(back_uv,grid_uv)
wrong_uv=project(Xc,K,R,t)
metrics['unprojection']={'roundtrip_max_px':float(np.max(np.abs(back_uv-grid_uv))), 'wrong_frame_mean_px':float(np.mean(np.linalg.norm(wrong_uv-grid_uv,axis=1)))}
print(metrics['unprojection'])

<!-- C29 -->
## N09 · Similarità e allineamento

**Prima dell’allineamento.** Gauge è la libertà di scala, rotazione e traslazione globali non identificata dalle sole proiezioni. Una similarità ha sette parametri indipendenti: 1 scala positiva, 3 rotazione e 3 traslazione. Richiede corrispondenze note; non trova da sola quali punti corrispondano.

Un secondo sistema restituisce Y=2 Qg X+tg. Stimare una similarità su cinque corrispondenze note, senza rumore. Il fit controlla il cambio di gauge, ma se usato sui punti di test ne usa anche la GT. Riportare errore grezzo e allineato e grado di libertà dell’allineamento.

In [ ]:
# C30
Qg=np.array([[0.,-1,0],[1,0,0],[0,0,1]]); tg=np.array([3.,-2,1.]); Y=2*(X@Qg.T)+tg
# Umeyama, mappa X verso Y, vettori riga
xc=X-X.mean(0); yc=Y-Y.mean(0)
U,D,Vt=np.linalg.svd(yc.T@xc/len(X)); S=np.eye(3); S[-1,-1]=np.linalg.det(U@Vt)
Qfit=U@S@Vt; scale=float(np.sum(D*np.diag(S))/np.mean(np.sum(xc*xc,axis=1))); trans=Y.mean(0)-scale*(Qfit@X.mean(0))
Yfit=scale*(X@Qfit.T)+trans
assert np.allclose(Yfit,Y) and np.isclose(scale,2)
metrics['similarity']={'raw_rmse':float(np.sqrt(np.mean(np.sum((X-Y)**2,axis=1)))),'aligned_rmse':float(np.sqrt(np.mean(np.sum((Y-Yfit)**2,axis=1)))),'scale':scale,'rotation':Qfit.tolist(),'translation':trans.tolist(),'uses_ground_truth':True}
print(json.dumps(metrics['similarity'],indent=2))

<!-- C31 -->
## N10 · Resize e intrinseci
Dimezzare l’immagine con convenzione pixel-center u′=(u+.5)/2−.5. Aggiornare K con la trasformazione A. Confrontare con l’errore prodotto da K non aggiornata. Il crop richiederebbe anche una traslazione del principal point.

In [ ]:
# C32
a=.5; A=np.array([[a,0,(a-1)/2],[0,a,(a-1)/2],[0,0,1.]])
Ksmall=A@K
uvsmall=project(X,Ksmall,R,t); expected=(uv+.5)*a-.5
assert np.allclose(uvsmall,expected)
metrics['resize']={'K_new':Ksmall.tolist(),'correct_max_px':float(np.max(np.abs(uvsmall-expected))), 'stale_K_mean_px':float(np.mean(np.linalg.norm(uv-expected,axis=1)))}
print('K nuova:',Ksmall); print(metrics['resize'])

<!-- C33 -->
## Metriche: un errore piccolo e una conclusione troppo grande

**Prima delle metriche.** MSE è la media dei quadrati degli errori fra intensità corrispondenti; PSNR significa Peak Signal-to-Noise Ratio e usa un logaritmo in base 10. MAX=1 per questi dati normalizzati in [0, 1]. SSIM confronta somiglianza strutturale: nessuna di queste metriche certifica distanze 3D.

Confrontare PSNR su una piccola immagine sintetica. Uno shift e un errore costante non misurano direttamente accuratezza metrica 3D. Fissare MAX=1 e MSE su tutti i pixel/canali.

In [ ]:
# C34
gt=np.linspace(0,1,64*64).reshape(64,64)
pred=np.clip(gt+.05,0,1)
mse=float(np.mean((pred-gt)**2)); psnr=float(10*np.log10(1/mse))
metrics['render_metric']={'mse':mse,'psnr_db':psnr,'MAX':1}
print(metrics['render_metric'])

<!-- C35 -->
## Geometry audit card e consegna
F01: dati e provenienza. F02: convenzioni di camera, scala e unità. F03: rendering e compositing con risultati. F04: due errori indotti e controlli che li individuano. F05: limiti e test successivo. Allegare JSON, figure e notebook compilato. Distinguere sempre dati sintetici, risultati degli autori e nuove inferenze locali.

In [ ]:
# C36
metrics['elapsed_cpu_seconds']=time.perf_counter()-START
(OUT/'risultati_lezione20.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
np.savez(OUT/'arrays_lezione20.npz',X=X,uv=uv,K=K,R=R,t=t,Sigma=Sigma,Sigma2=Sigma2,w=w)
manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='manifest.json'}
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
print('Salvati JSON, array, figure e hash in',OUT)

<!-- C37 -->
## Salvataggio locale e Colab
C38 crea uno ZIP dagli output della corsa corrente. In Colab avvia il download. Salvare separatamente il notebook con File > Scarica > .ipynb. Il pacchetto recupero mostra la corsa di riferimento salvata e non completa i tre TODO.

In [ ]:
# C38
archive=Path('risultati_lezione20.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): zf.write(p,arcname=p.name)
try:
    IN_COLAB=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB=False
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
print('Output pronto:',archive.resolve())

<!-- C39 -->
## Modelli completi, fuori dalla corsa CPU

**Richiamo dei modelli.** Positional encoding γ trasforma coordinate in seni/coseni fissati prima del MLP. SH sono armoniche sferiche, basi direzionali del colore; un quaternione normalizzato rappresenta una rotazione con quattro componenti vincolate. Occupancy indica appartenenza al volume, SDF distanza con segno. Queste rappresentazioni non sono la stessa quantità della densità di estinzione σ.

NeRF e 3DGS richiedono fotografie calibrate, dati di scena e ottimizzazione per scena. VGGT è feed-forward: il repository ufficiale propone il checkpoint VGGT-1B e un percorso GPU. Nessuno dei tre modelli viene scaricato/eseguito qui. Per una prova distinta seguire [NeRF](https://github.com/bmild/nerf), [3DGS](https://github.com/graphdeco-inria/gaussian-splatting), [VGGT](https://github.com/facebookresearch/vggt), fissare commit/checkpoint/licenza, scene, preprocessing, device e memoria. Esportare camere/depth/point maps e applicare l’audit già sperimentato. Per VGGT: extrinsics camera-from-world OpenCV, point maps nel riferimento della prima camera. Non usare tempi degli autori come tempi misurati sul proprio Colab.

<!-- C40 -->
## Discussione conclusiva
E01. Perché una nuova vista fotorealistica non certifica una lunghezza in metri?

E02. Se scambio ordine di due splat, quando posso aspettarmi lo stesso RGB?

E03. Perché il controllo depth → point map → stessa camera può dare errore zero anche con una depth errata?